In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from math import radians, sin, cos, sqrt, atan2

import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("finalTrain.csv")

In [4]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,14-02-2022,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


In [5]:
df.shape

(45584, 20)

In [6]:
df.columns

Index(['ID', 'Delivery_person_ID', 'Delivery_person_Age',
       'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Time_Orderd',
       'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density',
       'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle',
       'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45584 entries, 0 to 45583
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45584 non-null  object 
 1   Delivery_person_ID           45584 non-null  object 
 2   Delivery_person_Age          43730 non-null  float64
 3   Delivery_person_Ratings      43676 non-null  float64
 4   Restaurant_latitude          45584 non-null  float64
 5   Restaurant_longitude         45584 non-null  float64
 6   Delivery_location_latitude   45584 non-null  float64
 7   Delivery_location_longitude  45584 non-null  float64
 8   Order_Date                   45584 non-null  object 
 9   Time_Orderd                  43853 non-null  object 
 10  Time_Order_picked            45584 non-null  object 
 11  Weather_conditions           44968 non-null  object 
 12  Road_traffic_density         44983 non-null  object 
 13  Vehicle_conditio

In [8]:
df.isnull().sum()

ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weather_conditions              616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken (min)                  0
dtype: int64

In [9]:
for i in df.columns:
    print(f"no of unique values in {i} are: {df[i].nunique()}")
    print("-------------------------------------------------------------")
    if (df[i].nunique()) <8:
        print(f"Unique values are {df[i].unique()}")
        print("**************************************************************")

no of unique values in ID are: 45584
-------------------------------------------------------------
no of unique values in Delivery_person_ID are: 1320
-------------------------------------------------------------
no of unique values in Delivery_person_Age are: 22
-------------------------------------------------------------
no of unique values in Delivery_person_Ratings are: 28
-------------------------------------------------------------
no of unique values in Restaurant_latitude are: 657
-------------------------------------------------------------
no of unique values in Restaurant_longitude are: 518
-------------------------------------------------------------
no of unique values in Delivery_location_latitude are: 4373
-------------------------------------------------------------
no of unique values in Delivery_location_longitude are: 4373
-------------------------------------------------------------
no of unique values in Order_Date are: 44
-----------------------------------------

In [10]:
## creating some features with the help of our existing features

features=[]
dtypes=[]
count=[]
unique=[]
missing=[]
missing_percentage =[]

for i in df.columns:
    features.append(i)
    count.append(len(df[i]))
    unique.append(df[i].nunique())
    missing.append(df[i].isnull().sum())
    missing_percentage.append(df[i].isnull().sum()/df.shape[0]*100)
    
dataframe = pd.DataFrame({
    'feature':features,
    'count':count,
    'unique':unique,
    'missing_no':missing,
    'missing_percentage':missing_percentage
})
dataframe.set_index('feature')

,count,unique,missing_no,missing_percentage
feature,,,,
ID,45584,45584,0,0.000000
Delivery_person_ID,45584,1320,0,0.000000
Delivery_person_Age,45584,22,1854,4.067217
Delivery_person_Ratings,45584,28,1908,4.185679
Restaurant_latitude,45584,657,0,0.000000
Restaurant_longitude,45584,518,0,0.000000
Delivery_location_latitude,45584,4373,0,0.000000
Delivery_location_longitude,45584,4373,0,0.000000
Order_Date,45584,44,0,0.000000


In [11]:
drop= ['Delivery_person_Age','ID']
df.drop(drop,axis=1,inplace=True)

In [12]:
# Changing Dattime column in datetime

df['Order_Date']=pd.to_datetime(df['Order_Date'], format='mixed')


In [13]:
df['Order_Date']

0       2022-12-02
1       2022-02-13
2       2022-04-03
3       2022-02-13
4       2022-02-14
           ...    
45579   2022-03-24
45580   2022-02-16
45581   2022-11-03
45582   2022-07-03
45583   2022-02-03
Name: Order_Date, Length: 45584, dtype: datetime64[ns]

In [14]:
df['year']= df['Order_Date'].dt.year
df['month']=df['Order_Date'].dt.month
df['day']=df['Order_Date'].dt.day

In [15]:
# dropping order date
df.drop('Order_Date',axis=1,inplace=True)

In [16]:
df.dropna(subset=['Time_Orderd'],inplace=True)

In [17]:
df['Time_Orderd'] = df['Time_Orderd'].str.replace('.',':')

In [18]:
# convert the order time colum to a time data type
df['Time_Orderd'] = pd.to_datetime(df['Time_Orderd'], format='%H:%M', errors='coerce')
df['Time_ordered_Hour'] = df['Time_Orderd'].dt.hour
df['Time_ordered_Hour'] = df['Time_ordered_Hour'].astype('Int32')

In [19]:
# Time order picked 
df['Time_Order_picked'] = pd.to_datetime(df['Time_Order_picked'], format='%H:%M', errors='coerce')
df['Time_Order_Picked_Hour'] = df['Time_Order_picked'].dt.hour
df['Time_Order_Picked_Hour'] = df['Time_Order_Picked_Hour'].astype('Int32')
# minute
df['Time_Order_Picked_Min'] = df['Time_Order_picked'].dt.minute
df['Time_Order_Picked_Min'] = df['Time_Order_Picked_Min'].astype('Int32')

In [20]:
# extracting city from delivery persons id
df['Delivery_City']=df['Delivery_person_ID'].str.split('RES',expand=True)[0]


In [21]:
df.head(2)


,Delivery_person_ID,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,...,Festival,City,Time_taken (min),year,month,day,Time_ordered_Hour,Time_Order_Picked_Hour,Time_Order_Picked_Min,Delivery_City
0,DEHRES17DEL01,4.2,30.327968,78.046106,30.397968,78.116106,1900-01-01 21:55:00,1900-01-01 22:10:00,Fog,Jam,...,No,Metropolitian,46,2022,12,2,21,22,10,DEH
1,KOCRES16DEL01,4.7,10.003064,76.307589,10.043064,76.347589,1900-01-01 14:55:00,1900-01-01 15:05:00,Stormy,High,...,No,Metropolitian,23,2022,2,13,14,15,5,KOC


In [22]:
df['Delivery_City'].unique()

array(['DEH', 'KOC', 'PUNE', 'LUDH', 'KNP', 'MUM', 'MYS', 'HYD', 'KOL',
       'RANCHI', 'COIMB', 'CHEN', 'JAP', 'SUR', 'BANG', 'GOA', 'AURG',
       'AGR', 'VAD', 'ALH', 'BHP', 'INDO'], dtype=object)

In [23]:
rating_map = round(df.groupby('Delivery_person_ID')['Delivery_person_Ratings'].mean(),1)

In [24]:
# Filling null value with the mean of person id
rating_map=round(df.groupby('Delivery_person_ID')['Delivery_person_Ratings'].mean(),1).to_dict()
df['Delivery_person_Ratings']=df['Delivery_person_Ratings'].fillna(df['Delivery_person_ID'].map(rating_map))

In [25]:
df['Delivery_person_Ratings'].isnull().sum().sum()

np.int64(0)

In [ ]:
#! pip install folium

In [ ]:
# working with map
import folium
m=folium.Map(tiles='cartodb positron')
m